# 📈 SPY Stock Price Prediction with LSTM
Predict next-day SPY closing price using an LSTM neural network.

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import plotly.graph_objects as go

sys.path.insert(0, os.path.abspath('.'))
from src.fetch_data import fetch_spy_data
from src.preprocess import prepare_data, add_features
from src.lstm_model import build_model, train_model
from src.evaluate import compute_metrics, plot_predictions, plot_training_history, plot_residuals

## 1. Data Acquisition

In [ ]:
csv_path = 'data/spy_daily.csv'
if not os.path.exists(csv_path):
    df = fetch_spy_data(period='5y', save=True)
else:
    df = pd.read_csv(csv_path, index_col=0, parse_dates=True)

print(f'Shape: {df.shape}')
df.tail()

## 2. Exploratory Data Analysis

In [ ]:
fig = go.Figure()
fig.add_trace(go.Candlestick(
    x=df.index, open=df['Open'], high=df['High'],
    low=df['Low'], close=df['Close'], name='SPY'
))
fig.update_layout(title='SPY Daily OHLC (5 Years)', height=500,
                  xaxis_rangeslider_visible=False, template='plotly_dark')
fig.show()

In [ ]:
df_feat = add_features(df)
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=df_feat.index, y=df_feat['Close'], name='Close'))
fig2.add_trace(go.Scatter(x=df_feat.index, y=df_feat['MA_5'], name='MA 5'))
fig2.add_trace(go.Scatter(x=df_feat.index, y=df_feat['MA_20'], name='MA 20'))
fig2.update_layout(title='Close with Moving Averages', template='plotly_dark', height=400)
fig2.show()

## 3. Preprocessing & Sequence Creation

In [ ]:
SEQ_LEN = 60
X_train, y_train, X_test, y_test, scaler, close_scaler, df_proc = \
    prepare_data(df, seq_length=SEQ_LEN, train_ratio=0.8)

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}   y_test:  {y_test.shape}')
print(f'Features: Close, Volume, MA_5, MA_20, RSI, Returns')

## 4. Build & Train LSTM Model

In [ ]:
model = build_model(input_shape=(X_train.shape[1], X_train.shape[2]), units=64)
model.summary()

In [ ]:
history = train_model(model, X_train, y_train, X_test, y_test,
                      epochs=50, batch_size=32)

In [ ]:
fig_hist = plot_training_history(history)
fig_hist.show()

## 5. Predictions vs Actual

In [ ]:
y_pred_scaled = model.predict(X_test).flatten()
y_true_prices = close_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
y_pred_prices = close_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

split_idx = int(len(df_proc) * 0.8)
test_dates = df_proc.index[split_idx + SEQ_LEN : split_idx + SEQ_LEN + len(y_true_prices)]

fig_pred = plot_predictions(test_dates, y_true_prices, y_pred_prices)
fig_pred.show()

In [ ]:
fig_zoom = plot_predictions(
    test_dates[-60:], y_true_prices[-60:], y_pred_prices[-60:],
    title='Last 60 Days — Actual vs Predicted'
)
fig_zoom.show()

## 6. Evaluation Metrics

In [ ]:
metrics = compute_metrics(y_true_prices, y_pred_prices)
for k, v in metrics.items():
    print(f'{k:>20s}: {v}')

In [ ]:
fig_res = plot_residuals(y_true_prices, y_pred_prices)
fig_res.show()

## 7. Save Results

In [ ]:
results_df = pd.DataFrame({'Date': test_dates, 'Actual': y_true_prices, 'Predicted': y_pred_prices})
results_df.to_csv('data/predictions.csv', index=False)

with open('data/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Saved predictions.csv and metrics.json')